# Exercise preparation — Python, and the course plumbing

MSc Finance · Investments · FHNW · Autumn 2026

We work through this notebook together in class, in two parts.

**Part 1 — Python itself.** No data, no repository, no helpers. We build a small table by
hand, take it apart, loop over it and write one function. There is no finance in it worth
the name; the point is that you can read and write the six or seven constructs the rest of
the semester is made of.

**Part 2 — the plumbing.** The same six constructs, now on a century of real returns pulled
straight from the course repository, ending in a figure and a download.

Before you change anything: **File → Save a copy in Drive**. And whenever a result looks
odd, run **Runtime → Restart and run all** before doing anything else.

---
# Part 1 — Python itself

## 1 · Order matters, not position

A notebook is a list of cells, but the cell is not where anything lives. Names live in the
**runtime**, and a name exists from the moment its cell has *run* — not from the moment you
typed it. Scrolling past a cell does not run it.

Run the next cell three times in a row with `Shift+Enter` and watch the number change. The
cell does not change; the runtime does.

In [ ]:
counter = counter + 1 if "counter" in dir() else 1
print("this cell has now run", counter, "time(s)")

That is why almost every confusing result in this course is fixed by
**Runtime → Restart and run all**: it empties the runtime and re-executes everything from
the top, in the order it appears. If the notebook survives that, it works.

## 2 · Objects and names

`=` does not mean equals. It means *store the thing on the right under the name on the
left*. Everything in Python is an object of some type, and the type decides what you are
allowed to do with it. A `#` starts a comment; the last line of a cell displays itself
without needing `print`.

In [ ]:
x = 4                     # an integer
y = 2.5                   # a float
label = "Equity"          # a string

print(x + y)              # 6.5 — numbers add
print(label + " fund")    # strings join
print(type(x), type(y), type(label))

x + y                     # the last line displays on its own

`label + 1` would fail with `TypeError: can only concatenate str`. That is Python telling
you the type does not support the operation, not that something is broken.

## 3 · A vector: the Series

A **Series** is a column of numbers with a label on every row. It is the object you get
whenever you pull a single column out of a table, so it is worth meeting on its own first.

In [ ]:
import pandas as pd

r = pd.Series([0.12, -0.04, 0.08, 0.04],
              index=["Jan", "Feb", "Mar", "Apr"])
r

Two ways to reach one element, and one way to reach several. The `:.2%` inside the braces
is what turns an unreadable float into a percentage — §7 on your cheat sheet.

In [ ]:
print(r["Feb"])           # by label
print(r.iloc[1])          # by position — counting starts at 0
print(f"mean {r.mean():.2%}   sd {r.std():.2%}")   # a statistic is a method

r[r > 0]                  # a boolean filter: only the positive months

Note `r > 0`. It is not a question; it produces a Series of `True`/`False`, and putting
that inside the brackets keeps the rows where it is `True`. That single idea replaces most
of what a loop would otherwise have to do.

## 4 · A matrix: the DataFrame

A **DataFrame** is a table: several Series sharing one index. Four months, three assets,
monthly returns as decimals — `0.12` is 12 %.

In [ ]:
rets = pd.DataFrame({
    "Equity": [0.12, -0.04, 0.08, 0.04],
    "Bond":   [0.02,  0.00, 0.04, 0.02],
    "Cash":   [0.01,  0.01, 0.01, 0.01],
}, index=["Jan", "Feb", "Mar", "Apr"])

print(rets.shape)         # (rows, columns)
rets

Columns first, then rows. The distinction that catches everybody: **one name gives a Series
back, a list of names gives a DataFrame back**, even when the list has one entry in it.

In [ ]:
print(type(rets["Equity"]))      # Series
print(type(rets[["Equity"]]))    # DataFrame

rets[["Equity", "Bond"]]         # two columns

In [ ]:
print(rets.loc["Feb"])           # one row, by label

print(rets.loc["Feb":"Mar"])     # label slice — Mar IS included
print(rets.iloc[0:2])            # position slice — row 2 is NOT included

That asymmetry is deliberate in `pandas` and it is the single most common source of
off-by-one errors in this course. Label slicing includes the endpoint; position slicing
does not.

## 5 · Storing results in new objects

Nothing so far has changed `rets`. Each of those lines produced a new object and threw it
away. To keep one, give it a name.

In [ ]:
equity = rets["Equity"]                        # a Series
first_half = rets.iloc[0:2]                    # a DataFrame
excess = rets.sub(rets["Cash"], axis=0)        # every column minus Cash, row by row

print(rets.mean())                             # the mean of each column
print()
print(excess)

`axis=0` says *line the two up by row*. Without it `pandas` would try to match the Cash
labels against the column names and give you a table of `NaN`, which is the failure mode to
recognise: not an error message, just a table full of nothing.

## 6 · A loop

A loop repeats a block once per item. The indented lines are the block; the indentation is
the syntax, not decoration.

In [ ]:
for name in rets.columns:
    print(f"{name:8s} {rets[name].mean():6.2%}")

Loops earn their keep when each pass produces a row of a table. Collect one dictionary per
pass in a list, then hand the list to `pd.DataFrame` — this pattern builds almost every
table you will produce this semester.

In [ ]:
rows = []
for name in rets.columns:
    rows.append({"Asset": name,
                 "Mean": rets[name].mean(),
                 "Vol": rets[name].std()})

summary = pd.DataFrame(rows).set_index("Asset")
summary

## 7 · A function

Anything you compute more than twice belongs in a function: `def`, a name, the inputs in
brackets, and a `return` for whatever comes out.

In [ ]:
def annualise(monthly):
    """Turn a monthly return into the annual return it compounds to."""
    return (1 + monthly) ** 12 - 1

print(f"{annualise(0.05):.2%}")           # one number in, one number out
print()
print(annualise(rets.mean()))            # a Series in, a Series out

The same function, unchanged, worked on a single number and on a whole column. That is the
habit worth taking from Part 1: write the operation once, for one thing, and let `pandas`
apply it to everything.

Seven constructs, and that is the whole language for our purposes: a name, a Series, a
DataFrame, label and position indexing, a boolean filter, a loop, a function.

---
# Part 2 — Getting data, and getting results out

Same constructs, real data. Nothing new to learn here except where the data comes from and
how the results leave again.

## 8 · The opening ritual

One cell, run once per session. The `!wget` line downloads the shared helper module into
the runtime; the imports make the tools available; `setup_style()` applies the colours and
plotting defaults used in the lecture slides.

If the runtime disconnects, this cell has to be run again — that is what
*Restart and run all* does for you.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from exercise_utils import setup_style, load_returns, save_results
setup_style()

BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_01/data/histretSP_investments.xlsx"
SHEET = "Total Return Index_Damadoran"

print("Setup complete.")

## 9 · Load the data

`load_returns` reads an Excel file of total-return **index levels** straight from the
repository. Each series starts at 100 in 1927 and grows as if all income were reinvested.

In [ ]:
idx = load_returns(DATA_URL, sheet=SHEET)

print(idx.shape)          # (rows, columns)
print(list(idx.columns))  # exact column names — copy from here, do not retype
idx.head()

Those two lines are the diagnostics to reach for before asking anyone anything.
`print(x.shape)` answers *is it the size I think it is?* and `print(x.columns)` answers
*is the name spelled the way I typed it?* A `KeyError` is almost always the second one.

## 10 · From index levels to returns

An index level on its own says little. The annual return is the relative change from one
year to the next,

$$R_t = \frac{I_t}{I_{t-1}} - 1.$$

`load_returns(..., to_returns=True)` does exactly this and drops the first (empty) row.

In [ ]:
rets = load_returns(DATA_URL, sheet=SHEET, to_returns=True)

print(rets.shape)
rets.head()

Same object as in Part 1, ninety-eight rows instead of four. Everything you did to the toy
table works here unchanged: `rets["US T. Bond"]`, `rets.loc[1990:]`, `rets.mean()`.

## 11 · A first figure

A logarithmic vertical axis is the natural choice here: equal vertical distances then mean
equal *percentage* changes, which is what lets us compare series that differ enormously in
scale by the end of the sample.

In [ ]:
ASSETS = ["S&P 500 (includes dividends)", "US T. Bond", "3-month T.Bill"]

fig, ax = plt.subplots(figsize=(8, 4.5))
idx[ASSETS].plot(ax=ax, logy=True)
ax.set_xlabel("Year")
ax.set_ylabel("Index level, log scale (1927 = 100)")
plt.show()

## 12 · Take your results with you

The runtime forgets everything. `save_results` bundles figures and tables into one ZIP and
hands it to your browser. If nothing downloads, allow pop-ups for the Colab domain and run
the cell again.

In [ ]:
save_results(figures={"indices": fig},
             tables={"returns": rets}, name="warmup")

## What comes next

That is the whole workflow. This afternoon you use it on your own: descriptive statistics
for three asset classes, a century split into decades, a regression to read, one figure and
an export.

Two things about the afternoon. Tasks 1 to 3 are worked with the printed cheat sheet and
nothing else — no AI — so that you can later check the code an AI writes for you. The sheet
follows the sections of this notebook one for one, so anything you have just typed is on
it. And at the end of the session the solution notebook appears in Moodle.

Start from `ex01_skeleton.ipynb`, the second Colab link in Moodle.